In [1]:
import sksurv #scikit-survival
import pandas as pd
import numpy as np
import sys
import os


sys.path.append(os.path.abspath("../../"))
from src.utils.ConvertTextToCsv import TextToCsv
from src.utils.Preprocessing import Preprocessor
%load_ext autoreload
%autoreload 2

In [2]:
pp = Preprocessor()
df_clincal_data = pd.read_csv("../../data/raw/brca_tcga_pub2015_clinical_data.tsv", sep='\t')

list_df = pp.total_type_len_type_cancer(df_clincal_data)
df_clincal_data["Tumor-Cancer"] = list_df

df_mRNA_transformed = TextToCsv("../../data/raw/data_mrna_seq_v2_rsem.txt")

Luminal A: 330 - Total(%): 0.40
Luminal B: 81 - Total(%):0.10
HER2-enriched: 23 - Total(%):0.03
TNBC: 85 - Total(%)0.10 
UNK: 299 - Total(%) 0.37
Shape of the CSV: (20440, 819)


In [3]:
df_merged = pp.merge_datasets(df_clincal_data, df_mRNA_transformed)

In [12]:
binary_groups = []
for _, row in df_merged.iterrows():
    ER_status = row.get("ER Status By IHC", pd.NA)
    
    if (ER_status == "Negative"):
        binary_groups.append(0)
    
    elif (ER_status == "Positive"):
        binary_groups.append(1)
    
    else:
        binary_groups.append(-1)

In [13]:
print(len(binary_groups))
print(binary_groups)

df_merged["Binary ER group"] = binary_groups


817
[1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, -1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, -1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, -1, 1, -1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1

In [14]:
binary_groups_df = df_merged.loc[
    df_merged["Binary ER group"].isin([0, 1]),
    ["Binary ER group"] +list(df_merged.columns[1:20441])
]

binary_groups_df["Binary ER group"].unique()

array([1, 0])

In [15]:
binary_reduce_df = pp.elimnation_zeros(binary_groups_df, "Binary ER group")

Max of zeros per row in the dataset: 775
Avg of zeros per row in the dataset: 110.59877690802348
Median of zeros per row in the dataset: 0.0
Min of zeros per row in the dataset: 0
After the 0 elimination: 16264


In [16]:
print(f"Samples: {binary_reduce_df.shape[0]}, Genes: {binary_reduce_df.shape[1]}")

Samples: 775, Genes: 16264


In [ ]:
results_df, design, _ = pp.initialize_limma(binary_reduce_df, "Binary ER group")
results_df.rename(columns={"column0" : "Binary 0",
                           "column1": "Binary 1"})

,Binary 0,Binary 1,AveExpr,F,pvalue,adj_pvalue
10190,7.587871,7.643881,7.631233,58628.740884,0.000000e+00,0.000000e+00
10191,8.912371,8.022297,8.223281,20972.390177,0.000000e+00,0.000000e+00
56,9.820627,11.004093,10.736859,46571.716726,0.000000e+00,0.000000e+00
10194,2.864254,3.434416,3.305669,3742.256511,0.000000e+00,0.000000e+00
58,7.734712,8.049891,7.978722,19519.111418,0.000000e+00,0.000000e+00
...,...,...,...,...,...,...
4988,2.404297,2.061032,2.138543,419.623971,5.749432e-183,5.750846e-183
17494,1.954852,2.102671,2.069292,376.615675,2.740884e-164,2.741389e-164
5630,1.656050,1.954234,1.886902,367.969250,1.559500e-160,1.559692e-160
16977,1.783533,3.149988,2.841434,359.632269,6.511637e-157,6.512038e-157
